# Vectorized DLR Calculation

In [1]:
import numpy as np
from scipy import stats
import xarray as xr
import pandas as pd
import os
from netCDF4 import Dataset,num2date 
import time
import matplotlib.pyplot as plt
import cdsapi
import time
import pypsa

In [2]:
# functions from IEEE standard for calculating ampacity
# e# at the end of each function definition indicates which equation in the document it's referencing

# trig functions

def sind(x):
    return np.sin(np.deg2rad(x))
def cosd(x):
    return np.cos(np.deg2rad(x))
def tand(x):
    return np.tan(np.deg2rad(x))
def asind(x):
    return np.degrees(np.arcsin(x))
def acosd(x):
    return np.degrees(np.arccos(x))
def atand(x):
    return np.degrees(np.arctan(x))

# ampacity calc

def calc_I_e2(q_c,q_r,q_s,r):
    # inputs: convection heat loss (q_c), radiated heat loss (q_r), heat gain from sun (q_s)
    # output: ampacity (rated current)
    diff = (q_c + q_r - q_s)/r
    return np.sqrt(diff)

# convection equations

def calc_q_c_e6(q_cn, q_c1, q_c2):
    return np.maximum(np.maximum(q_cn, q_c1), q_c2)

def calc_q_cn_e7(rho_f,d_0,t_s,t_a):
    return 3.645*rho_f**0.5*d_0**0.75*(t_s-t_a)**1.25

def calc_q_c1_e9(k_ang,n_re,k_f,t_s,t_a):
    return k_ang*(1.01+1.35*n_re**0.52)*k_f*(t_s-t_a)

def calc_q_c2_e10(k_ang,n_re,k_f,t_s,t_a):
    return k_ang*0.754*n_re**0.6*k_f*(t_s-t_a)

def calc_n_re_e11(d_0,rho_f,v_w,mu_f):
    return (d_0*rho_f*v_w)/mu_f

def calc_t_film_e12(t_s,t_a):
    return (t_s + t_a)/2

def calc_mu_f_e13(t_film):
    return (1.458*10**(-6)*(t_film+273)**1.5)/(t_film + 383.4)

def calc_rho_f_e15(h_e,t_film):
    return (1.293-1.525*10**(-4)*h_e+6.379*10**(-9)*h_e**2)/(1+0.00367*t_film)

def calc_k_f_e17(t_film):
    return 2.424*10**(-2)+7.477*10**(-5)*t_film - 4.407*10**(-9)*t_film**2

def calc_k_ang_e19(phi):
    return 1.194 - cosd(phi) + 0.194*cosd(2*phi) + 0.368*sind(2*phi)

def calc_q_c_MAIN(d_0,t_s,t_a,v_w,phi,h_e):

    t_film = calc_t_film_e12(t_s,t_a)

    rho_f = calc_rho_f_e15(h_e,t_film)
    k_ang = calc_k_ang_e19(phi)
    mu_f = calc_mu_f_e13(t_film)
    k_f = calc_k_f_e17(t_film)

    n_re = calc_n_re_e11(d_0,rho_f,v_w,mu_f)
    
    q_cn = calc_q_cn_e7(rho_f,d_0,t_s,t_a)
    q_c1 = calc_q_c1_e9(k_ang,n_re,k_f,t_s,t_a)
    q_c2 = calc_q_c2_e10(k_ang,n_re,k_f,t_s,t_a)

    q_c = calc_q_c_e6(q_cn,q_c1,q_c2)

    return q_c

# radiated heat loss equations

def calc_q_r_e21(d_0,ep,t_s,t_a):
    return 17.8*d_0*ep*(((t_s+273)/100)**4 - ((t_a+273)/100)**4)

# solar heat gain

def calc_q_s_e23(alpha,q_se,theta,a_prime):
    return alpha*q_se*sind(theta)*a_prime

def calc_theta_e24(h_c,z_c,z_l):
    return acosd(cosd(h_c)*cosd(z_c-z_l))

def calc_h_c_e25(lat,delta,omega):
    return asind(cosd(lat)*cosd(delta)*cosd(omega)+sind(lat)*sind(delta))

def calc_omega_e26(hour):
    return (hour - 12)*15

def calc_delta_e27(day_of_year):
    return 23.45*sind((284+day_of_year)/365*360)

def calc_z_c_e28(chi, omega):
    c = np.zeros_like(omega)
    
    # omega < 0
    mask1 = omega < 0
    c[mask1 & (chi >= 0)] = 0
    c[mask1 & (chi < 0)] = 180
    
    # omega >= 0
    mask2 = omega >= 0
    c[mask2 & (chi >= 0)] = 180
    c[mask2 & (chi < 0)] = 360
    
    return c + atand(chi)

def calc_chi_e29(omega,lat,delta):
    return sind(omega)/(sind(lat)*cosd(omega)-cosd(lat)*tand(delta))

def calc_big_q_s_e30(coeff,h_c): # coeff - tbl 4
    Q_s = 0
    for i in range(7):
        Q_s = Q_s + coeff[i]*h_c**i
    Q_s = np.maximum(Q_s,0)
    return Q_s

def calc_q_se(big_q_s,h_e):
    k_solar = 1 + (1.148*10**(-4))*h_e + (-1.108*10**(-8))*h_e**2
    return k_solar*big_q_s

def calc_q_s_MAIN(alpha,a_prime,lat,hour,day_of_year,h_e,z_l,coeff,use_q,given_q_se):

    omega = calc_omega_e26(hour)
    delta = calc_delta_e27(day_of_year)
    chi = calc_chi_e29(omega,lat,delta)

    z_c = calc_z_c_e28(chi,omega)

    h_c = calc_h_c_e25(lat,delta,omega)

    theta = calc_theta_e24(h_c,z_c,z_l)
    big_q_s = calc_big_q_s_e30(coeff,h_c)

    q_se = calc_q_se(big_q_s,h_e)
    if use_q:
        q_s = calc_q_s_e23(alpha,given_q_se,theta,a_prime)
    else:
        q_s = calc_q_s_e23(alpha,q_se,theta,a_prime)

    return q_s

# resistance calc

def calc_R(R_high,R_low,T_high,T_low,T_avg):
    return ((R_high - R_low)/(T_high - T_low))*(T_avg - T_low) + R_low

# main boi

def calc_ampacity_MAIN(const,line,by_hour,use_q):
    r = calc_R(const["R_high"],const["R_low"],const["T_high"],const["T_low"],const["T_avg"])
    q_c = calc_q_c_MAIN(const["d_0"],const["t_s"],by_hour["t_a"],by_hour["v_w"],by_hour["phi"],line["h_e"])
    q_r = calc_q_r_e21(const["d_0"],const["ep"],const["t_s"],by_hour["t_a"])
    q_s = calc_q_s_MAIN(const["alpha"],const["d_0"],line["lat"],by_hour["hour"],by_hour["day_of_year"],line["h_e"],line["z_l"],const["coeff"],use_q,by_hour["q_se"])
    amp = calc_I_e2(q_c,q_r,q_s,r)
    
    return amp

In [3]:
solar_coeff_data = [
    -42.2391,
    63.8044,
    -1.9220,
    3.46921e-2,
    -3.61118e-4,
    1.94318e-6,
    -4.07608e-9
]

"""
const = {
    "R_high": 8.688*10**-5,
    "R_low": 7.283*10**-5,
    "T_high": 75,
    "T_low": 25,
    "T_avg": 100,
    "ep": 0.8,
    "alpha": 0.8,
    "t_s": 100,
    "d_0": 0.02814,
    "coeff": solar_coeff_data
}
"""

'\nconst = {\n    "R_high": 8.688*10**-5,\n    "R_low": 7.283*10**-5,\n    "T_high": 75,\n    "T_low": 25,\n    "T_avg": 100,\n    "ep": 0.8,\n    "alpha": 0.8,\n    "t_s": 100,\n    "d_0": 0.02814,\n    "coeff": solar_coeff_data\n}\n'

In [4]:
# load bus data
bus = pd.read_csv("mchjones/dlr_weather_data/decadal_comparison/complete_elevation_reference.csv",index_col=0,header=0)
bus.index = bus.index.astype(str)

In [5]:
print(bus.head())

             lat      lon   elev
2010001  48.2414 -124.578  136.0
2010002  47.6956 -124.184  280.0
2010003  47.0400 -124.057    6.0
2010004  46.9275 -124.172    5.0
2010005  46.9275 -124.172    5.0


## Apply

In [6]:
def load_weather(year,gcm,source):
    var = ['t','u','v','q']
    wus_data = {}
    start = time.time()
    for v in var:
        print(f"loading {v}")
        if source == "WUS":
            if gcm == "taiesm1":
                path = f"mchjones/dlr_weather_data/decadal_comparison/WUS_raw-weather_bus_good-time/{v}_{year}_WUS-TAIESM_base-dc-bus_WECC_time-fixed.csv"
            else:
                path = f"mchjones/dlr_weather_data/decadal_comparison/WUS_raw-weather_bus_good-time/{v}_{year}_WUS-{gcm}_base-dc-bus_WECC_time-fixed.csv"
        elif source == "ERA5":
            path = f"mchjones/dlr_weather_data/decadal_comparison/ERA5_raw-weather_bus/{v}_{year}_ERA5_base-dc-bus_WECC.csv"
        w = pd.read_csv(path,header=0,index_col=0)
        
        w.index = pd.to_datetime(w.index)
        w = w[~((w.index.month == 2) & (w.index.day == 29))]
        hours = np.arange(0,8760,1)
        w.index=hours
        w.columns = w.columns.astype(str)
        
        wus_data[v] = w
        
        if v == "q" and source == "ERA5":
            wus_data[v] = wus_data[v]/(60*60)
    print(f"Data for {year} loaded in {time.time() - start:.2f} s")
    return wus_data

def load_weather_aar(year,gcm,source):
    var = ['t','q']
    wus_data = {}
    start = time.time()
    for v in var:
        print(f"loading {v}")
        if source == "WUS":
            if gcm == "taiesm1":
                path = f"mchjones/dlr_weather_data/decadal_comparison/WUS_raw-weather_bus_good-time/{v}_{year}_WUS-TAIESM_base-dc-bus_WECC_time-fixed.csv"
            else:
                path = f"mchjones/dlr_weather_data/decadal_comparison/WUS_raw-weather_bus_good-time/{v}_{year}_WUS-{gcm}_base-dc-bus_WECC_time-fixed.csv"
        elif source == "ERA5":
            path = f"mchjones/dlr_weather_data/decadal_comparison/ERA5_raw-weather_bus/{v}_{year}_ERA5_base-dc-bus_WECC.csv"
        w = pd.read_csv(path,header=0,index_col=0)
        
        w.index = pd.to_datetime(w.index)
        w = w[~((w.index.month == 2) & (w.index.day == 29))]
        hours = np.arange(0,8760,1)
        w.index=hours
        w.columns = w.columns.astype(str)
        
        wus_data[v] = w
        
        if v == "q" and source == "ERA5":
            wus_data[v] = wus_data[v]/(60*60)
    print(f"Data for {year} loaded in {time.time() - start:.2f} s")
    return wus_data

def get_bus_by_hour(wus_data,bus):
    bus_data = {
        't': wus_data['t'][bus],
        'u': wus_data['u'][bus],
        'v': wus_data['v'][bus],
        'q': wus_data['q'][bus]
    }

    v_w = np.sqrt(bus_data['u']**2 + bus_data['v']**2) #CHANGE
    theta = np.degrees(np.arctan2(bus_data['v'], bus_data['u']))
    phi = np.minimum(
        np.abs((theta - 90 + 180) % 360 - 180),  # angle from +90°
        np.abs((theta + 90 + 180) % 360 - 180)   # angle from -90°
    )
    phi = phi.clip(lower=20, upper=70)
    
    t = bus_data['t'].values - 273.15

    by_hour = {
        "v_w": v_w.values/3,
        "t_a": t,
        "hour": np.arange(1,8761,1),
        "day_of_year": np.repeat(np.arange(1, 366), 24),
        "phi": phi, #np.zeros(8760), #45*np.ones(8760), #, # , # , # const phi 90*np.ones(8760)
        "q_se": bus_data['q'].values #CHANGE
    }
    
    return by_hour

def get_aar_wind_from_t(temp,t99,q):
    t_delta = temp - t99
    wind = np.where(t_delta > -8, 0.5, 0.4)
    wind = np.where(q < 1, 0, wind)    
    return wind

def get_bus_by_hour_aar(temp,bus,t99,q,approach):
    
    if approach == "540":
        wind = get_aar_wind_from_t(temp[bus].values - 273.15,t99[bus],q[bus])
    elif approach == "6":
        wind = 0.61*np.ones(8760)\

    by_hour = {
        "v_w": wind,
        "t_a": temp[bus].values - 273.15,
        "hour": np.arange(1,8761,1),
        "day_of_year": np.repeat(np.arange(1, 366), 24),
        "phi": 90*np.ones(8760), # const phi CHANGED THIS TO 0 FROM 90
        "q_se": 1000
    }
    
    return by_hour

def get_bus_line(row):
    line = {
        "z_l": 90,
        "lat": row["lat"],
        "h_e": row["elev"]
    }
    return line

def calc_system_dlrs(buses,data,use_q):
    num_buses = len(buses) 
    dlr = pd.DataFrame(index=np.arange(0,8760,1),columns=buses.index)
    start = time.time()
    i = 0
    print("Starting DLR calculations...")
    for idx, row in buses.iterrows():
        bus_by_hour = get_bus_by_hour(data,idx)
        line = get_bus_line(row)
        dlr[idx] = calc_ampacity_MAIN(const,line,bus_by_hour,use_q)
        
        i += 1
        if i % 1000 == 0:
            print(f"{i/num_buses*100:.1f}% done! (walltime: {time.time() - start:.2f})")
    print("DLR calculation complete")
    return dlr

def calc_system_aars(buses,temp,use_q,t99,q,approach):
    num_buses = len(buses) 
    dlr = pd.DataFrame(index=np.arange(0,8760,1),columns=buses.index)
    start = time.time()
    i = 0
    print("Starting AAR calculations...")
    for idx, row in buses.iterrows():
        bus_by_hour = get_bus_by_hour_aar(temp,idx,t99,q,approach)
        line = get_bus_line(row)
        dlr[idx] = calc_ampacity_MAIN(const,line,bus_by_hour,use_q)
        
        i += 1
        if i % 1000 == 0:
            print(f"{i/num_buses*100:.1f}% done! (walltime: {time.time() - start:.2f})")
    print("DLR calculation complete")
    return dlr

def calc_system_baseline(buses,bus_by_hour,use_q):
    baseline = pd.DataFrame(index=buses.index,columns=["baseline"])
    for idx, row in buses.iterrows():
        line = get_bus_line(row)
        baseline.loc[idx,"baseline"] = calc_ampacity_MAIN(const,line,bus_by_hour,use_q)
    baseline = baseline.squeeze()
    return baseline

def trunk(df):
    cap = 1.3
    trunk = df.where(df <= cap,cap)
    print(f"Dataframe truncated at {cap}")
    return trunk

In [7]:
def calc_system_baseline_given_t(buses,bus_by_hour,t_99,use_q):
    baseline = pd.DataFrame(index=buses.index,columns=["baseline"])
    for idx, row in buses.iterrows():
        line = get_bus_line(row)
        bus_by_hour["t_a"] = t_99.loc[idx]
        baseline.loc[idx,"baseline"] = calc_ampacity_MAIN(const,line,bus_by_hour,use_q)
    baseline = baseline.squeeze()
    return baseline

In [8]:
def check_exist(path):
    if os.path.exists(path):
        return True
    else:
        return False

In [9]:
def get_cond(conductor):
    if conductor == "drake":
        const = {
            "R_high": 8.688*10**-5,
            "R_low": 7.283*10**-5,
            "T_high": 75,
            "T_low": 25,
            "T_avg": 75,
            "ep": 0.7,
            "alpha": 0.9,
            "t_s": 75,
            "d_0": 0.02814,
            "coeff": solar_coeff_data
        }
    elif conductor == "condor":
        const = {
            "R_high": 0.0000892388451443569,
            "R_low": 0.0000733318899868673,
            "T_high": 75,
            "T_low": 20,
            "T_avg": 75,
            "ep": 0.8,
            "alpha": 0.8,
            "t_s": 75,
            "d_0": 0.02773678502,
            "coeff": solar_coeff_data
        }
    elif conductor == "martin":
        const = {
            "R_high": 0.0000534776902887139,
            "R_low": 0.0000453248689183715,
            "T_high": 75,
            "T_low": 20,
            "T_avg": 75,
            "ep": 0.8,
            "alpha": 0.8,
            "t_s": 75,
            "d_0": 0.03616958047,
            "coeff": solar_coeff_data
        }
    elif conductor == "cardinal":
        const = {
            "R_high": 0.0000748031496062992,
            "R_low": 0.0000617879796204326,
            "T_high": 75,
            "T_low": 20,
            "T_avg": 75,
            "ep": 0.8,
            "alpha": 0.8,
            "t_s": 75,
            "d_0": 0.0303783836,
            "coeff": solar_coeff_data
        }
    else:
        print("Invalid conductor type!")
    return const

# Round 3 Calculations
Using 
- T_a = 25 C
- T_s = T_film = T_avg = 75 C
- Conductor: Drake
- Emissivity: 0.7
- Absorptivity: 0.9
- Length cap: 100 km
- AAR: $\phi$ = 90, $v_w$ = 0.61 m/s
- DLR: $v_w/3$, $20 < \phi < 70$

In [10]:
const = get_cond("drake")

### SLR-Adjusted

In [54]:
path = "mchjones/dlr_weather_data/decadal_comparison/round3/line_discount.csv"
discount = pd.read_csv(path,header=0,index_col=0)

In [11]:
gcms = ["ec3","ec3veg","miroc6","mpi","taiesm1"]

In [14]:
def get_worse_slr(x,y):
    return min(x, y)

def get_line_from_bus_slr(bus_dlrs,line_info):
    line_dlrs = pd.Series(index=line_info.index)
    i = 0
    n = len(line_info)
    start = time.time()
    for line, row in line_info.iterrows():
        key_start = time.time()
        
        b0 = bus_dlrs[row["bus0"]]
        b1 = bus_dlrs[row["bus1"]]
        line_dlrs[line] = get_worse_slr(b0,b1)
        
        i = i + 1
        if i % (n // 20) == 0 or i == n:
            check = time.time()
            print(f'{(i/n)*100:.3g}% done! (line time: {(check - key_start):.2f}, walltime: {(check - start):.2f})')
    return line_dlrs

def load_nc(path):
    network = xr.open_dataset(path)
    return network

def get_line_info(pypsa_results):
    line_info = pd.DataFrame({
        'bus0': pypsa_results['lines_bus0'],
        'bus1': pypsa_results['lines_bus1']
    }, index=pypsa_results['lines_i'].values)
    return line_info

base_path = "mchjones/pypsa-usa/workflow/resources/test_clustering/western/elec_base_network.nc"
n = load_nc(base_path)
line_info = get_line_info(n)

In [17]:
baseline_by_hour = {
    "v_w": 0.61,
    "t_a": 40,
    "hour": 11,
    "day_of_year": 161,
    "phi": 90,
    "q_se": 1000
}
base_25 = calc_system_baseline_given_t(bus,baseline_by_hour,ref_25,True)
line_25 = get_line_from_bus_slr(base_25,line_info)    

5% done! (line time: 0.00, walltime: 0.03)
9.99% done! (line time: 0.00, walltime: 0.06)
15% done! (line time: 0.00, walltime: 0.08)
20% done! (line time: 0.00, walltime: 0.11)
25% done! (line time: 0.00, walltime: 0.14)
30% done! (line time: 0.00, walltime: 0.16)
35% done! (line time: 0.00, walltime: 0.19)
40% done! (line time: 0.00, walltime: 0.22)
45% done! (line time: 0.00, walltime: 0.24)
50% done! (line time: 0.00, walltime: 0.27)
55% done! (line time: 0.00, walltime: 0.30)
60% done! (line time: 0.00, walltime: 0.32)
65% done! (line time: 0.00, walltime: 0.35)
70% done! (line time: 0.00, walltime: 0.38)
75% done! (line time: 0.00, walltime: 0.40)
80% done! (line time: 0.00, walltime: 0.43)
84.9% done! (line time: 0.00, walltime: 0.46)
89.9% done! (line time: 0.00, walltime: 0.49)
94.9% done! (line time: 0.00, walltime: 0.51)
99.9% done! (line time: 0.00, walltime: 0.54)
100% done! (line time: 0.00, walltime: 0.54)


In [18]:
slr_adj = pd.DataFrame(index=line_info.index,columns=gcms)
for gcm in gcms:
    print(f"\nStarting {gcm}")
    t_99 = pd.read_csv(f"mchjones/dlr_weather_data/t99_WUS-{gcm}_base-dc-bus_WECC_2015-2024.csv",header=0,index_col=0)
    t_99_C = t_99 - 273.15
    ref_temp_2024 = t_99_C.loc[2024]
    ref_25 = pd.Series(25, index=ref_temp_2024.index)

    base_gcm = calc_system_baseline_given_t(bus,baseline_by_hour,ref_temp_2024,True)
    save_path = f"mchjones/dlr_weather_data/decadal_comparison/round3/bus_adj_{gcm}.csv"
    base_gcm.to_csv(save_path)
    print(f"Bus baseline exported to {save_path}")
    
    line_dlrs = get_line_from_bus_slr(base_gcm,line_info)
    save_path = f"mchjones/dlr_weather_data/decadal_comparison/round3/line_adj_{gcm}.csv"
    line_dlrs.to_csv(save_path)
    print(f"Line baseline exported to {save_path}")
    
    slr_adj[gcm] = line_dlrs / line_25


Starting ec3
Bus baseline exported to /nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/bus_adj_ec3.csv
5% done! (line time: 0.00, walltime: 0.02)
9.99% done! (line time: 0.00, walltime: 0.05)
15% done! (line time: 0.00, walltime: 0.07)
20% done! (line time: 0.00, walltime: 0.10)
25% done! (line time: 0.00, walltime: 0.12)
30% done! (line time: 0.00, walltime: 0.14)
35% done! (line time: 0.00, walltime: 0.17)
40% done! (line time: 0.00, walltime: 0.19)
45% done! (line time: 0.00, walltime: 0.22)
50% done! (line time: 0.00, walltime: 0.24)
55% done! (line time: 0.00, walltime: 0.26)
60% done! (line time: 0.00, walltime: 0.29)
65% done! (line time: 0.00, walltime: 0.31)
70% done! (line time: 0.00, walltime: 0.34)
75% done! (line time: 0.00, walltime: 0.36)
80% done! (line time: 0.00, walltime: 0.39)
84.9% done! (line time: 0.00, walltime: 0.41)
89.9% done! (line time: 0.00, walltime: 0.44)
94.9% done! (line time: 0.00, walltime: 0.46)
99.9% done! (line 

## Calculate Bus DLRs
### Baseline standard conductor SLR calculation

In [11]:
gcms = ["ec3"]#,"ec3veg","miroc6","mpi","taiesm1"]
years = np.arange(2045,2055,1) #np.arange(2045,2055,1) [2050]
source = "WUS"

In [16]:
t_99_hist = pd.read_csv("/home/mchjones/climate588/project/data/bus_t_99.csv",header=0,index_col=0)
t_99_hist_C = t_99_hist - 273.15
ref_temp = t_99_hist_C.loc[2000]
ref_25 = pd.Series(25, index=ref_temp.index)

In [13]:
baseline_by_hour = {
    "v_w": 0.61,
    "t_a": 40,
    "hour": 11,
    "day_of_year": 161,
    "phi": 90,
    "q_se": 1000
}
print(const)
baseline_w_t = calc_system_baseline_given_t(bus,baseline_by_hour,ref_25,True)

{'R_high': 8.688000000000001e-05, 'R_low': 7.283e-05, 'T_high': 75, 'T_low': 25, 'T_avg': 75, 'ep': 0.7, 'alpha': 0.9, 't_s': 75, 'd_0': 0.02814, 'coeff': [-42.2391, 63.8044, -1.922, 0.0346921, -0.000361118, 1.94318e-06, -4.07608e-09]}


### Raw AAR and DLR Calculations

In [14]:
for gcm in gcms:
    print(f"\nStarting {gcm}\n===============================")
    start = time.time()
    for year in years:
        print(f"\nStaring {year}")
        year_start = time.time()
        aar_wind = "6" #540 or 6
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/aar_{year}_{source}-{gcm}_base-dc-bus_{aar_wind}.csv"
        check = check_exist(save_path)
        if check:
            print(f"{gcm} {year} already exists ({save_path})")
        else:
            data = load_weather_aar(year,gcm,source)
            aar = calc_system_aars(bus,data['t'],True,ref_25,data['q'],aar_wind)
            aar = aar.round(3)
            aar.to_csv(save_path)
            print(f"Exported to {save_path} ({time.time()-year_start:.2f} s)")
    print(f"GCM {gcm} completed ({(time.time()-start)/60:.2f} m)")


Starting ec3

Staring 2045
ec3 2045 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/aar_2045_WUS-ec3_base-dc-bus_6.csv)

Staring 2046
ec3 2046 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/aar_2046_WUS-ec3_base-dc-bus_6.csv)

Staring 2047
ec3 2047 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/aar_2047_WUS-ec3_base-dc-bus_6.csv)

Staring 2048
ec3 2048 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/aar_2048_WUS-ec3_base-dc-bus_6.csv)

Staring 2049
ec3 2049 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/aar_2049_WUS-ec3_base-dc-bus_6.csv)

Staring 2050
ec3 2050 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/aar_2050_WUS-ec3_base-dc-bus_6.csv)

Staring 2051
ec3 2051 already e

In [15]:
for gcm in gcms:
    print(f"\nStarting {gcm}\n===============================")
    start = time.time()
    for year in years:
        print(f"\nStaring {year}")
        year_start = time.time()
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/dlr_{year}_{source}-{gcm}_base-dc-bus_phi20-70-div3.csv"
        check = check_exist(save_path)
        if check:
            print(f"{gcm} {year} already exists ({save_path})")
        else:
            data = load_weather(year,gcm,source)
            dlr = calc_system_dlrs(bus,data,True)
            dlr = dlr.round(3)
            dlr.to_csv(save_path)   
            print(f"Exported to {save_path} ({time.time()-year_start:.2f} s)")
    print(f"GCM {gcm} completed ({(time.time()-start)/60:.2f} m)")


Starting ec3

Staring 2045
ec3 2045 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/dlr_2045_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Staring 2046
ec3 2046 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/dlr_2046_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Staring 2047
ec3 2047 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/dlr_2047_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Staring 2048
ec3 2048 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/dlr_2048_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Staring 2049
ec3 2049 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/dlr_2049_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Staring 2050
ec3 2050 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/dlr_2050

### Convert to relative

In [16]:
# AAR

for gcm in gcms:
    print(f"\nStarting {gcm}\n===============================")
    start = time.time()
    for year in years:
        print(f"\nStarting relative conversion for {year}")
        year_start = time.time()
        aar_wind = "6"
        load_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/aar_{year}_{source}-{gcm}_base-dc-bus_{aar_wind}.csv"
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_{year}_{source}-{gcm}_base-dc-bus_{aar_wind}.csv"
        check = check_exist(save_path)
        if check:
            print(f"{gcm} {year} already exists ({save_path})")
        else:
            aar = pd.read_csv(load_path,header=0,index_col=0)
            print(f"Raw AAR imported from {load_path}")
            rel = aar.div(baseline_w_t,axis=1)
            rel = rel.round(3)
            rel.to_csv(save_path)
            print(f"Relative AAR exported to {save_path} ({time.time()-year_start:.2f} s)")
    print(f"GCM {gcm} completed ({(time.time()-start)/60:.2f} m)")


Starting ec3

Starting relative conversion for 2045
ec3 2045 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2045_WUS-ec3_base-dc-bus_6.csv)

Starting relative conversion for 2046
ec3 2046 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2046_WUS-ec3_base-dc-bus_6.csv)

Starting relative conversion for 2047
ec3 2047 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2047_WUS-ec3_base-dc-bus_6.csv)

Starting relative conversion for 2048
ec3 2048 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2048_WUS-ec3_base-dc-bus_6.csv)

Starting relative conversion for 2049
ec3 2049 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2049_WUS-ec3_base-dc-bus_6.csv)

Starting relative conversion for 2050
ec3 205

In [17]:
# DLR

for gcm in gcms:
    print(f"\nStarting {gcm}\n===============================")
    start = time.time()
    for year in years:
        print(f"\nStarting relative conversion for {year}")
        year_start = time.time()
        load_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/dlr_{year}_{source}-{gcm}_base-dc-bus_phi20-70-div3.csv"
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_{year}_{source}-{gcm}_base-dc-bus_phi20-70-div3.csv"
        check = check_exist(save_path)
        if check:
            print(f"{gcm} {year} already exists ({save_path})")
        else:
            dlr = pd.read_csv(load_path,header=0,index_col=0)
            print(f"Raw DLR imported from {load_path}")
            rel = dlr.div(baseline_w_t,axis=1)
            rel = rel.round(3)
            rel.to_csv(save_path)   
            print(f"Exported to {save_path} ({time.time()-year_start:.2f} s)")
    print(f"GCM {gcm} completed ({(time.time()-start)/60:.2f} m)")


Starting ec3

Starting relative conversion for 2045
ec3 2045 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2045_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Starting relative conversion for 2046
ec3 2046 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2046_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Starting relative conversion for 2047
ec3 2047 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2047_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Starting relative conversion for 2048
ec3 2048 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2048_WUS-ec3_base-dc-bus_phi20-70-div3.csv)

Starting relative conversion for 2049
ec3 2049 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2049_WUS-ec3_base-dc-bus_phi20

### Check distributions

In [12]:
def below(df,value):
    count = (df < value).sum().sum()
    p = count/(df.shape[0]*df.shape[1])*100
    return p

In [15]:
for gcm in gcms:
    for year in years:
        aar_wind = 6
        print(f"Distribution analysis for {gcm}, {year}\n================================================")
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_{year}_{source}-{gcm}_base-dc-bus_{aar_wind}.csv"
        aar = pd.read_csv(save_path,header=0,index_col=0)
        print(f"Relative AAR imported from {save_path}")
        
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_{year}_{source}-{gcm}_base-dc-bus_phi20-70-div3.csv"
        dlr = pd.read_csv(save_path,header=0,index_col=0)
        print(f"Relative DLR imported from {save_path}")
        
        p = below(aar,1)
        print(f"AAR < SLR: {p:.2f}%")
        p = below(dlr,1)
        print(f"DLR < SLR: {p:.2f}%")
        p = below(dlr,aar)
        print(f"DLR < AAR: {p:.2f}%")

Distribution analysis for ec3, 2050
Relative AAR imported from /nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2050_WUS-ec3_base-dc-bus_6.csv
Relative DLR imported from /nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/sensitivity/rel-dlr_2050_WUS-ec3_base-dc-bus_phi20-70-div3.csv
AAR < SLR: 12.80%
DLR < SLR: 9.49%
DLR < AAR: 29.85%
Distribution analysis for ec3veg, 2050
Relative AAR imported from /nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2050_WUS-ec3veg_base-dc-bus_6.csv
Relative DLR imported from /nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/sensitivity/rel-dlr_2050_WUS-ec3veg_base-dc-bus_phi20-70-div3.csv
AAR < SLR: 10.82%
DLR < SLR: 8.48%
DLR < AAR: 29.79%
Distribution analysis for miroc6, 2050
Relative AAR imported from /nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2050_WUS-miroc6_base-

## Bus DLRs > Line DLRs

In [32]:
def load_nc(path):
    network = xr.open_dataset(path)
    return network

def get_bus(pypsa_results):
    coord = pd.DataFrame({
        'lon': pypsa_results['buses_x'],
        'lat': pypsa_results['buses_y']
    }, index=pypsa_results['buses_i'].values)
    return coord

def get_line_info(pypsa_results):
    line_info = pd.DataFrame({
        'bus0': pypsa_results['lines_bus0'],
        'bus1': pypsa_results['lines_bus1']
    }, index=pypsa_results['lines_i'].values)
    return line_info

def get_worse(dlr0,dlr1):
    return [min(x, y) for x, y in zip(dlr0, dlr1)]

def get_line_from_bus(bus_dlrs,line_info):
    line_dlrs = pd.DataFrame(index=bus_dlrs.index,columns=line_info.index)
    i = 0
    n = len(line_info)
    start = time.time()
    for line, row in line_info.iterrows():
        key_start = time.time()
        
        b0 = bus_dlrs[row["bus0"]]
        b1 = bus_dlrs[row["bus1"]]
        line_dlrs[line] = get_worse(b0,b1)
        
        i = i + 1
        if i % (n // 20) == 0 or i == n:
            check = time.time()
            print(f'{(i/n)*100:.3g}% done! (line time: {(check - key_start):.2f}, walltime: {(check - start):.2f})')
    return line_dlrs

In [33]:
base_path = "//mchjones/pypsa-usa/workflow/resources/test_clustering/western/elec_base_network.nc"
n = load_nc(base_path)
bus = get_bus(n)
line_info = get_line_info(n)

In [20]:
for gcm in gcms:
    for year in years:
        print(f"\nCalculating line AARs  for {gcm} {year}")
        start = time.time()
        aar_wind = 6
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_{year}_{source}-{gcm}_base-dc-line_{aar_wind}.csv"
        load_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_{year}_{source}-{gcm}_base-dc-bus_{aar_wind}.csv"
        
        bus_dlrs = pd.read_csv(load_path,header=0,index_col=0)
        print("Bus AARs loaded")

        line_dlrs = get_line_from_bus(bus_dlrs,line_info)
        line_dlrs = line_dlrs.round(3)
        line_dlrs.to_csv(save_path)
        print(f"get_line_from_bus ran in {time.time() - start:.2f} s")
        print(f"Exported to {save_path}")


Calculating line AARs  for ec3 2045
Bus AARs loaded
5% done! (line time: 0.00, walltime: 1.32)
9.99% done! (line time: 0.00, walltime: 2.64)
15% done! (line time: 0.00, walltime: 3.97)
20% done! (line time: 0.00, walltime: 5.51)
25% done! (line time: 0.00, walltime: 6.85)
30% done! (line time: 0.00, walltime: 8.19)
35% done! (line time: 0.00, walltime: 9.53)
40% done! (line time: 0.00, walltime: 10.86)
45% done! (line time: 0.00, walltime: 12.21)
50% done! (line time: 0.00, walltime: 13.56)
55% done! (line time: 0.00, walltime: 14.90)
60% done! (line time: 0.00, walltime: 16.24)
65% done! (line time: 0.00, walltime: 17.59)
70% done! (line time: 0.00, walltime: 18.94)
75% done! (line time: 0.00, walltime: 20.29)
80% done! (line time: 0.00, walltime: 21.79)
84.9% done! (line time: 0.00, walltime: 23.16)
89.9% done! (line time: 0.00, walltime: 24.54)
94.9% done! (line time: 0.00, walltime: 25.93)
99.9% done! (line time: 0.00, walltime: 27.32)
100% done! (line time: 0.17, walltime: 27.50)

In [21]:
for gcm in gcms:
    for year in years:
        print(f"\nCalculating line DLRs for {gcm} {year}")
        start = time.time()
        
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_{year}_{source}-{gcm}_base-dc-line_phi20-70-div3.csv"
        load_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_{year}_{source}-{gcm}_base-dc-bus_phi20-70-div3.csv"
        
        bus_dlrs = pd.read_csv(load_path,header=0,index_col=0)
        print("Bus DLRs loaded")

        line_dlrs = get_line_from_bus(bus_dlrs,line_info)
        line_dlrs = line_dlrs.round(3)
        line_dlrs.to_csv(save_path)
        print(f"get_line_from_bus ran in {time.time() - start:.2f} s")
        print(f"Exported to {save_path}")


Calculating line DLRs for ec3 2045
Bus DLRs loaded
5% done! (line time: 0.00, walltime: 1.32)
9.99% done! (line time: 0.00, walltime: 2.65)
15% done! (line time: 0.00, walltime: 3.99)
20% done! (line time: 0.00, walltime: 5.33)
25% done! (line time: 0.00, walltime: 6.67)
30% done! (line time: 0.00, walltime: 8.01)
35% done! (line time: 0.00, walltime: 9.37)
40% done! (line time: 0.00, walltime: 10.72)
45% done! (line time: 0.00, walltime: 12.07)
50% done! (line time: 0.00, walltime: 13.43)
55% done! (line time: 0.00, walltime: 14.80)
60% done! (line time: 0.00, walltime: 16.16)
65% done! (line time: 0.00, walltime: 17.53)
70% done! (line time: 0.00, walltime: 18.90)
75% done! (line time: 0.00, walltime: 20.27)
80% done! (line time: 0.00, walltime: 21.65)
84.9% done! (line time: 0.00, walltime: 23.03)
89.9% done! (line time: 0.00, walltime: 24.42)
94.9% done! (line time: 0.00, walltime: 25.85)
99.9% done! (line time: 0.00, walltime: 27.36)
100% done! (line time: 0.17, walltime: 27.54)


## Apply length cap

In [14]:
base_path = "//mchjones/pypsa-usa/workflow/resources/test_clustering/western/elec_base_network.nc"
n = pypsa.Network(base_path)
lines = n.lines

INFO:pypsa.io:Imported network elec_base_network.nc has buses, lines, line_types, links, transformers


In [15]:
length = lines["length"]
long = length[length > 100]
long_lines = list(long.index)
print(long_lines)

['88487', '88824', '88866', '88876', '88900', '89133', '89144', '89163', '89199', '89200', '89208', '89209', '89216', '89251', '89354', '89421', '89503', '89536', '89560', '89634', '89668', '90120', '90121', '90122', '90130', '90152', '90153', '90154', '90174', '90178', '90216', '90231', '90259', '90285', '90316', '90317', '90318', '90319', '90320', '90346', '90348', '90358', '90359', '90360', '90389', '90390', '90506', '90528', '90596', '90598', '90877', '90896', '90912', '90913', '90923', '90940', '91009', '91010', '91025', '91026', '91027', '91038', '91080', '91136', '91919', '92296', '92474', '92476', '92536', '92584', '92622', '92623', '92669', '92708', '94134', '94146', '94247', '95133', '95268', '95301', '95305', '95306', '95322', '95473', '95538', '95742', '95743', '95757', '95765', '95769', '95776', '95779', '95787', '95998', '95999', '96009', '96016', '96065', '96079', '96130', '96149', '96151', '96179', '96662', '96671', '96907', '96919', '97176', '97217', '97218', '97231', 

In [16]:
for gcm in gcms:
    for year in years:
        print(f"\nApplying line cap for {gcm} {year}")
        start = time.time()
        
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_{year}_{source}-{gcm}_base-dc-line_6_cap100.csv"
        load_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_{year}_{source}-{gcm}_base-dc-line_6.csv"
        
        check = check_exist(save_path)
        if check:
            print(f"{gcm} {year} already exists ({save_path})")
        else:
            aar = pd.read_csv(load_path,header=0,index_col=0)
            print("AARs loaded")
            matched = [x for x in long_lines if x in aar.columns]
            missing = [x for x in long_lines if x not in aar.columns]
            print(missing)

            cap_aar = aar.copy()
            cap_aar[long_lines] = 1
            cap_aar.to_csv(save_path)
            print(f"100 km cap applied ({time.time() - start:.2f} s)")
            print(f"Exported to {save_path}")


Applying line cap for ec3 2045
ec3 2045 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2045_WUS-ec3_base-dc-line_6_cap100.csv)

Applying line cap for ec3 2046
ec3 2046 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2046_WUS-ec3_base-dc-line_6_cap100.csv)

Applying line cap for ec3 2047
ec3 2047 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2047_WUS-ec3_base-dc-line_6_cap100.csv)

Applying line cap for ec3 2048
ec3 2048 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2048_WUS-ec3_base-dc-line_6_cap100.csv)

Applying line cap for ec3 2049
ec3 2049 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-aar_2049_WUS-ec3_base-dc-line_6_cap100.csv)

Applying line cap for ec3 2050
ec3 2050 already exists

In [17]:
for gcm in gcms:
    for year in years:
        print(f"\nApplying line cap for {gcm} {year}")
        start = time.time()
        
        save_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_{year}_{source}-{gcm}_base-dc-line_phi20-70-div3_cap100.csv"
        load_path = f"//mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_{year}_{source}-{gcm}_base-dc-line_phi20-70-div3.csv"
        check = check_exist(save_path)
        if check:
            print(f"{gcm} {year} already exists ({save_path})")
        else:
            dlr = pd.read_csv(load_path,header=0,index_col=0)
            print("DLRs loaded")
            matched = [x for x in long_lines if x in dlr.columns]
            missing = [x for x in long_lines if x not in dlr.columns]
            print(missing)

            cap_dlr = dlr.copy()
            cap_dlr[long_lines] = 1
            cap_dlr.to_csv(save_path)
            print(f"100 km cap applied ({time.time() - start:.2f} s)")
            print(f"Exported to {save_path}")


Applying line cap for ec3 2045
ec3 2045 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2045_WUS-ec3_base-dc-line_phi20-70-div3_cap100.csv)

Applying line cap for ec3 2046
ec3 2046 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2046_WUS-ec3_base-dc-line_phi20-70-div3_cap100.csv)

Applying line cap for ec3 2047
ec3 2047 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2047_WUS-ec3_base-dc-line_phi20-70-div3_cap100.csv)

Applying line cap for ec3 2048
ec3 2048 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2048_WUS-ec3_base-dc-line_phi20-70-div3_cap100.csv)

Applying line cap for ec3 2049
ec3 2049 already exists (/nfs/turbo/seas-mtcraig-climate/mchjones/dlr_weather_data/decadal_comparison/round3/rel-dlr_2049_WUS-ec3_base-dc-line_phi20-70-div3_cap100.